In [1]:
from pathlib import Path
import shutil, random, csv
from config import config

In [2]:
root = Path("../../..")

In [3]:
SOURCE_DIR = Path(r"D:\MSc\3. Spring 2026\CSE715\Project\wavs3sec") # -> DO NOT REMOVE
# SOURCE_DIR = Path(input("Enter the path where the above Kaggle dataset was downloaded in your device: "))
DEST_DIR = root / config.RAW_AUDIO_DIR_BN
METADATA_DIR = root / config.METADATA_DIR
METADATA_FILE = METADATA_DIR / "metadata_bn.csv"

In [4]:
SOURCE_DIR.exists(), METADATA_DIR.exists() # has to be true

(True, True)

In [5]:
DEST_DIR.mkdir(exist_ok=True, parents=True)

In [6]:
def find_audio_files(subdir):
    return [f for f in subdir.iterdir() if f.is_file() and f.suffix.lower() == ".wav"]

In [7]:
random.seed(42)

In [8]:
metadata_rows = []

In [9]:
from math import ceil

In [10]:
N_SAMPLES = ceil(config.N_SUBSET / 2)

In [11]:
all_available_files = []
for subdir in SOURCE_DIR.iterdir():
    if subdir.is_dir():
        files = find_audio_files(subdir=subdir)
        for f in files:
            all_available_files.append((f, subdir))

In [12]:
sampled_entries = random.sample(all_available_files, k=N_SAMPLES)

In [13]:
metadata_rows = []
for src_file, subdir in sampled_entries:
    dst_file = DEST_DIR / src_file.name
    # handle duplicate names
    if dst_file.exists():
        counter = 1
        while True:
            candidate = DEST_DIR / f"{src_file.stem}_{counter}{src_file.suffix}"
            if not candidate.exists():
                dst_file = candidate
                break
            counter += 1
    shutil.copy2(src_file, dst_file)
    metadata_rows.append({"label": dst_file.name, "genre": subdir.name})

In [14]:
with open(METADATA_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["label", "genre"])
    writer.writeheader()
    writer.writerows(metadata_rows)